### Pre-Processing OPM data

How to use: 

1. run in osld enviroment - will need to 'pip install PyQt6' 'pip install pyqt6 pyqt6-tools' 
2. put in your data 
3. look at the psds to determine bad channels
4. unmark bad channels 
5. go through data and mark/unmark bad segments 
6. close mne interactive figure 

In [1]:
#Importing necessary libraries
import os
os.chdir("/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks") #sets working directory to the repo, so that all imports work correctly
print(os.getcwd())
import numpy as np
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import mne
# matplotlib.use('Qt5Agg')
#matplotlib.use('qtagg')
# print(matplotlib.get_backend())
from osl_dynamics.meeg import amm, preproc

import pyqtgraph as pg
# Disable OpenGL in PyQtGraph to prevent Linux GPU crashes
pg.setConfigOption('useOpenGL', False)
mne.viz.set_browser_backend('qt')
%matplotlib qt


/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks
Using qt as 2D backend.


In [2]:
def preprocess_opm(rawfile, preprocfile, figuresdir):

    ''' 
    1.Detects bad channels automatically and also allows for manual editing of selected bad channels. 
    2.detects bad segments and allows for review and modification (adding/removing) of the bad segments detected.
    3.Then saves out a fif file with the bad channels and segments annotated.
    '''

    #1. LOAD IN THE RAW DATA 
    print('opening raw file')
    raw = mne.io.read_raw_fif(rawfile, preload=True) #loading in the raw .fif data
    print('done reading in raw data')

    # --------------------------------------------------------------------------------------------------------------

    #2. PRE-PROCESSING
    #raw = raw.set_channel_types({'Rig': 'stim', 'Lef': 'stim'})
    #raw, amm_info = amm.apply_amm(raw)
    raw = raw.resample(sfreq=250)
    raw = raw.filter(l_freq=1, h_freq=120, method="iir", iir_params={"order": 5, "ftype": "butter"})
    raw = raw.notch_filter([50, 100])

    # --------------------------------------------------------------------------------------------------------------

    #3. CROPPING THE DATASET TO -1s BEFORE THE FIRST EVENT AND +20s AFTER THE LAST EVENT
    channels = ['Rig', 'Lef']
    all_events = []
    for ch in channels:
        events = mne.find_events(raw, stim_channel=ch, shortest_event=1)
        if events.size > 0:
            all_events.append(events)
            
    all_events = np.vstack(all_events)
    # Get the first and last sample across both channels
    first_sample = all_events[:, 0].min()
    last_sample = all_events[:, 0].max()
    # Add 20 seconds after last event
    sfreq = raw.info['sfreq']
    extra_samples_1s = int(1 * sfreq)
    extra_samples_20s = int(20 * sfreq)
    # Crop
    max_time = raw.times[-1]
    tmin = (first_sample - extra_samples_1s) / sfreq
    tmax = (last_sample + extra_samples_20s) / sfreq

    tmax = min(tmax, max_time)

    raw = raw.crop(tmin=tmin, tmax=tmax)

    # --------------------------------------------------------------------------------------------------------------

    #4. DETECT BAD CHANNELS 
    #raw = preproc.detect_bad_channels(raw, picks="mag", significance_level=0.1)
    #fig1 = preproc.plot_channel_stds(raw)
    #fig2 = raw.plot_psd()
    fig3 = raw.compute_psd().plot()
    plt.show(block=True)
    
    # --------------------------------------------------------------------------------------------------------------

    #5. DETECT BAD SEGMENTS
    raw = preproc.detect_bad_segments(raw, picks="mag", significance_level=0.1)
    raw = preproc.detect_bad_segments(raw, picks="mag", mode="diff", significance_level=0.1)
    raw = preproc.detect_bad_segments(raw, picks="mag", metric="kurtosis", significance_level=0.1)
    raw.plot(block=True, picks=['meg'], n_channels=192, duration=15)

    # --------------------------------------------------------------------------------------------------------------

    #6. SAVE PREPROCESSED DATA 
    raw = preproc.decimate_headshape_points(raw)
    raw.save(preprocfile, overwrite=True)

    #Sav preproc info 
    fig4 = raw.compute_psd().plot(show=False)  # do not show yet
    fig4.savefig(f"{figuresdir}/psd_markedbadchannels.png", dpi=600) 
    plt.close(fig4)  # optional, closes figure to free memory
    fig5 = raw.plot_psd(show=False)  # do not show yet
    fig5.savefig(f"{figuresdir}/psd_nobadchannels.png", dpi=600) 
    plt.close(fig5)  # optional, closes figure to free memory
    

    return None

In [4]:
subjects = ['016']
sessions = ["002"]
tasks = ["braille"]
runs = ["001"]


base = "/rdrives/DRS-foundation-brain/zoe_data/BIDS"
deriv = f"{base}/derivatives_anna_01042026"
preprocessed_dir = f"{deriv}/preprocessed"
os.makedirs(deriv, exist_ok=True)

for subject in subjects:
    for session in sessions:
        for task in tasks:
            for run in runs:

                id = f"sub-{subject}_ses-{session}_task-{task}_run-{run}"
                raw_file = f"{base}/sub-{subject}/ses-{session}/meg/{id}_meg.fif" 
                preproc_dir = f"{preprocessed_dir}/{id}"
                os.makedirs(preproc_dir, exist_ok=True)
                preproc_file = f"{preproc_dir}/{id}_preproc-raw.fif" 
                figures_dir = f"{preproc_dir}/figures"
                os.makedirs(figures_dir, exist_ok=True)

                

                preprocess_opm(raw_file, preproc_file, figures_dir)


opening raw file
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/sub-016/ses-002/meg/sub-016_ses-002_task-braille_run-001_meg.fif...
    Range : 0 ... 506239 =      0.000 ...  1349.971 secs
Ready.
Reading 0 ... 506239  =      0.000 ...  1349.971 secs...
done reading in raw data
Finding events on: A1, bra-0, A3, Rig, A5, A6, A7, A8, bra-1, A10, bra-2, A12, Lef, A14, Bra, A16, T1, key, T3, T4, T5, T6, T7, T8, T9, T10, T11
12832 events found on stim channel bra-0
Event IDs: [2]
40 events found on stim channel Rig
Event IDs: [3]
802 events found on stim channel bra-1
Event IDs: [2]
665 events found on stim channel bra-2
Event IDs: [2]
40 events found on stim channel Lef
Event IDs: [3]
400 events found on stim channel Bra
Event IDs: [3]
139 events found on stim channel key
Event IDs: [1]
Trigger channel T6 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
Trigger channel T8 has a non-zero initial value of 1 (consider using initial_e

/tmp/ipykernel_593757/2866753787.py:19: RuntimeWarning: Some events are duplicated in your different stim channels. 11 events were ignored during deduplication.
  raw = raw.resample(sfreq=250)


Finding events on: A1, bra-0, A3, Rig, A5, A6, A7, A8, bra-1, A10, bra-2, A12, Lef, A14, Bra, A16, T1, key, T3, T4, T5, T6, T7, T8, T9, T10, T11
11944 events found on stim channel bra-0
Event IDs: [2]
40 events found on stim channel Rig
Event IDs: [3]
765 events found on stim channel bra-1
Event IDs: [2]
659 events found on stim channel bra-2
Event IDs: [2]
40 events found on stim channel Lef
Event IDs: [3]
380 events found on stim channel Bra
Event IDs: [3]
133 events found on stim channel key
Event IDs: [1]
Trigger channel T6 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
Trigger channel T8 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
Trigger channel T9 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
Trigger channel T11 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
Filtering raw data in 1 contiguous segment
S

/tmp/ipykernel_593757/2866753787.py:19: RuntimeWarning: Some events are duplicated in your different stim channels. 25 events were ignored during deduplication.
  raw = raw.resample(sfreq=250)
/tmp/ipykernel_593757/2866753787.py:19: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  raw = raw.resample(sfreq=250)


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 1651 samples (6.604 s)

Finding events on: Rig
40 events found on stim channel Rig
Event IDs: [3]
Finding events on: Lef
40 events found on stim channel Lef
Event IDs: [3]
Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).

Bad segment detection
---------------------
Modality: mag
Mode: None
Metric: std
Significance level: 0.1
Maximum fraction: 0.1
Found 52 bad segments: 130.0/1307.2 seconds rejected (9.9%)

Bad segment detection
---------------------
Omitting 32500 of 326810 (9.94%) samples, retaining 294310 (90.06%) samples.
Modality: mag
Mode: diff
Metric: std
Significanc

qt.qpa.wayland: There are no outputs - creating placeholder screen
